# Validating Unstructured Data with GX Core

In the first notebook we validated a **table** (NYC taxi trips). But a lot of ML data is **unstructured**: images, audio, text documents...

Great Expectations validates tables, so how can we use it here? The trick is to **turn every file into a row of features** that describe it, and then validate that table:

```
image files  ──►  feature extraction  ──►  features table (DataFrame)  ──►  GX validation
 cat.1.jpg         width, height,           one row per file                same workflow as
 dog.7.jpg         color mode, hash, ...                                    the tabular lab
```

We will learn how to:

- compute **features** from a folder of images
- write **expectations** about those features (e.g. "every file is a readable RGB JPEG of at least 64 px, with no duplicates")
- validate a clean dataset, then a **corrupted** one, and see what GX catches

In [ ]:
import hashlib
import shutil
import urllib.request
import zipfile
from pathlib import Path

import great_expectations as gx
import pandas as pd
from PIL import Image, ImageStat

## Importing the data

We use the **cats vs. dogs** image dataset from the [DVC tutorials](https://dvc.org/doc/start/data-management/data-versioning) (41 MB): 1000 training and 800 validation JPEG images, in one folder per class.

With DVC installed, you would download it with:

```bash
dvc get https://github.com/treeverse/dataset-registry tutorials/versioning/data.zip
```

To avoid an extra dependency, we download the same file straight from DVC's public storage. The cell skips the download if the data is already there.

In [ ]:
DATA_URL = "https://remote.dvc.org/docs/dvc/files/md5/fa/9c0eb4173d86695b4e800219651360"
DATA_DIR = Path("data")
zip_path = DATA_DIR / "data.zip"

if not (DATA_DIR / "train").exists():
    DATA_DIR.mkdir(exist_ok=True)
    print("Downloading data.zip...")
    urllib.request.urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(".")  # the archive contains a top-level data/ folder

for folder in sorted(DATA_DIR.glob("*/*")):
    print(folder, len(list(folder.glob("*"))), "files")

## From files to features

For every image we compute a few **features** (also called *metadata* or *descriptors*):

| Feature | What it tells us |
|---|---|
| `label` | the class, taken from the parent folder (`cats` / `dogs`) |
| `extension`, `file_size_kb` | basic file properties |
| `is_readable` | can the image be fully decoded, or is the file corrupt? |
| `format` | the *actual* encoding (JPEG, PNG, ...), read from the file content and not from the extension |
| `mode` | color mode: `RGB` (color), `L` (grayscale), `RGBA` (with transparency), ... |
| `width`, `height`, `aspect_ratio` | image size and shape |
| `mean_brightness` | mean pixel intensity (0 = black, 255 = white) |
| `md5` | hash of the file content, which shows exact duplicates |

If a file cannot be decoded, its image features stay empty (`None`).

In [ ]:
def extract_features(path: Path) -> dict:
    """Compute a row of features describing one image file."""
    features = {
        "file_path": str(path),
        "label": path.parent.name,
        "extension": path.suffix.lower(),
        "file_size_kb": path.stat().st_size / 1024,
        "md5": hashlib.md5(path.read_bytes()).hexdigest(),
        "is_readable": False,
        "format": None,
        "mode": None,
        "width": None,
        "height": None,
        "aspect_ratio": None,
        "mean_brightness": None,
    }
    try:
        with Image.open(path) as img:
            img.load()  # fully decode the pixels: fails on truncated/corrupt files
            features.update(
                is_readable=True,
                format=img.format,
                mode=img.mode,
                width=img.width,
                height=img.height,
                aspect_ratio=img.width / img.height,
                mean_brightness=ImageStat.Stat(img.convert("L")).mean[0],
            )
    except OSError:
        pass  # leave the image features empty
    return features


def build_feature_table(folder: Path) -> pd.DataFrame:
    """One row per file found (recursively) in `folder`."""
    files = sorted(p for p in folder.rglob("*") if p.is_file())
    return pd.DataFrame([extract_features(p) for p in files])

Let's build the features table for the **training set**.

In [ ]:
df = build_feature_table(DATA_DIR / "train")
df.head()

Before writing expectations, **look at the data**: the summary statistics help us choose sensible thresholds.

In [ ]:
df.describe()

In [ ]:
for column in ["label", "extension", "format", "mode", "is_readable"]:
    print(df[column].value_counts(dropna=False).to_dict())

print("duplicate files:", df["md5"].duplicated().sum())

The training set looks clean: 500 cats and 500 dogs, all readable RGB JPEGs, at least 80 px per side, and no duplicates.

From now on, the workflow is **the same as in the tabular lab**.

### Create a **Data Context** and connect to the features table

We reuse the same File Data Context (the `gx/` folder). We add a separate Data Source, so this notebook doesn't change the objects created by the first one.

In [ ]:
context = gx.get_context(mode="file")

data_source = context.data_sources.add_or_update_pandas("pandas images")
data_asset = data_source.add_dataframe_asset(name="image features")
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch definition")

### Create an **Expectation Suite** for images

Each expectation is about a feature column, but it describes a property of the **image files**.

> **Note:** most column expectations *ignore missing values*. An unreadable file has `format = None`, so it would never fail the `format` expectation. That is why we check `is_readable` explicitly.

The aspect-ratio expectation uses the `mostly` parameter: real datasets contain a few legitimate outliers (very wide panoramas, tall portraits), so we accept the batch if **at least 99%** of the rows satisfy it.

In [ ]:
suite = context.suites.add_or_update(gx.ExpectationSuite(name="Image dataset expectations"))

expectations = [
    # Dataset level: we expect roughly 1000 images (no missing or partial upload)
    gx.expectations.ExpectTableRowCountToBeBetween(min_value=900, max_value=1100),
    # Only the two known classes
    gx.expectations.ExpectColumnDistinctValuesToEqualSet(
        column="label", value_set=["cats", "dogs"]
    ),
    # No corrupt or truncated files
    gx.expectations.ExpectColumnValuesToBeInSet(column="is_readable", value_set=[True]),
    # Files are really JPEGs (not just named .jpg)
    gx.expectations.ExpectColumnValuesToBeInSet(column="format", value_set=["JPEG"]),
    # Color images only (no grayscale, no transparency)
    gx.expectations.ExpectColumnValuesToBeInSet(column="mode", value_set=["RGB"]),
    # No tiny thumbnails: the model needs at least 64x64 pixels
    gx.expectations.ExpectColumnValuesToBeBetween(column="width", min_value=64),
    gx.expectations.ExpectColumnValuesToBeBetween(column="height", min_value=64),
    # Reasonable shapes, tolerating a few outliers
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="aspect_ratio", min_value=0.5, max_value=2.0, mostly=0.99
    ),
    # No (almost) completely black or white images
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="mean_brightness", min_value=10, max_value=245
    ),
    # No duplicate images: duplicates bias training and can leak into the test set
    gx.expectations.ExpectColumnValuesToBeUnique(column="md5"),
]

for expectation in expectations:
    suite.add_expectation(expectation)

### Create a **Validation Definition** and a **Checkpoint**

As in the first notebook, the Checkpoint updates the Data Docs after each run.

We also set a `result_format` that reports the `file_path` of the rows that fail an expectation. When a feature is out of range, we want to know **which image** to look at.

In [ ]:
validation_definition = context.validation_definitions.add_or_update(
    gx.ValidationDefinition(
        name="Image validation definition", data=batch_definition, suite=suite
    )
)

checkpoint = context.checkpoints.add_or_update(
    gx.Checkpoint(
        name="image checkpoint",
        validation_definitions=[validation_definition],
        actions=[gx.checkpoint.actions.UpdateDataDocsAction(name="update_data_docs")],
        result_format={
            "result_format": "COMPLETE",
            "unexpected_index_column_names": ["file_path"],
        },
    )
)

A small helper prints one line per expectation and, for failures, the files that caused them.

In [ ]:
def summarize(checkpoint_result):
    print("SUCCESS" if checkpoint_result.success else "FAILURE", "\n")
    for run in checkpoint_result.run_results.values():
        for r in run.results:
            cfg = r.expectation_config
            status = "✅" if r.success else "❌"
            print(f"{status} {cfg.type}({cfg.kwargs.get('column', '')})")
            unexpected = r.result.get("unexpected_index_list") or []
            if unexpected and r.success:
                print("      tolerated by `mostly`:")
            for row in unexpected:
                print("      ", row["file_path"])

### Validate the clean training set

In [ ]:
checkpoint_result = checkpoint.run(batch_parameters={"dataframe": df})
summarize(checkpoint_result)

## Simulating a bad data drop

Imagine that a new batch of images arrives from a scraper, a labeling vendor or another team. Let's make a copy of the training set and "break" it in a few realistic ways:

| Problem | How we create it |
|---|---|
| grayscale image | convert a cat to black & white |
| thumbnail | resize a dog to 16×16 pixels |
| duplicate | copy an existing image under a new name |
| truncated download | keep only the first 30% of the bytes of a JPEG |
| black frame | a completely black image |
| wrong format | a PNG file saved with a `.jpg` extension |

In [ ]:
CORRUPTED_DIR = DATA_DIR / "train_corrupted"
shutil.rmtree(CORRUPTED_DIR, ignore_errors=True)
shutil.copytree(DATA_DIR / "train", CORRUPTED_DIR)

cats, dogs = CORRUPTED_DIR / "cats", CORRUPTED_DIR / "dogs"

# 1. grayscale image
with Image.open(cats / "cat.1.jpg") as img:
    img.convert("L").save(cats / "cat.grayscale.jpg")

# 2. tiny thumbnail
with Image.open(dogs / "dog.1.jpg") as img:
    img.resize((16, 16)).save(dogs / "dog.thumbnail.jpg")

# 3. exact duplicate
shutil.copy(cats / "cat.2.jpg", cats / "cat.copy.jpg")

# 4. truncated (corrupt) file
data = (dogs / "dog.2.jpg").read_bytes()
(dogs / "dog.truncated.jpg").write_bytes(data[: len(data) * 3 // 10])

# 5. all-black image
Image.new("RGB", (400, 300), color="black").save(dogs / "dog.black.jpg")

# 6. PNG disguised as a JPEG
with Image.open(cats / "cat.3.jpg") as img:
    img.save(cats / "cat.png_inside.jpg", format="PNG")

In [ ]:
df_corrupted = build_feature_table(CORRUPTED_DIR)
df_corrupted[df_corrupted["file_path"].str.contains(r"\.(?:grayscale|thumbnail|copy|truncated|black|png_inside)\.")]

### Validate the new batch

We run **the same Checkpoint** with the new features table. The expectations stay the same, only the data changes.

In [ ]:
checkpoint_result = checkpoint.run(batch_parameters={"dataframe": df_corrupted})
summarize(checkpoint_result)

Every injected problem is caught by at least one expectation. Note that:

- the **duplicate** fails `md5` uniqueness. GX lists *both* copies, the original and the new one;
- the **truncated** file fails only `is_readable`. Its other features are empty, and the column expectations skip them;
- `ExpectTableRowCountToBeBetween` and `aspect_ratio` (thanks to `mostly`) still pass.

### Explore the **Data Docs**

The Checkpoint has updated the Data Docs site, which now also contains the image suite and its two validation runs.

In [ ]:
context.open_data_docs()

In the Data Docs, open the `Image dataset expectations` suite.

**Exercises**

1. Validate the `data/validation` folder with the same Checkpoint. Does it pass? Should the row-count expectation be different for it?
2. Add a new **feature** and an **expectation** for it.
3. **Data leakage**: build one features table for `train` and `validation` together (with a `split` column) and check that no image appears in both splits.

## Beyond images

The same pattern works for any unstructured data: **compute features, then validate the table**.

| Data | Example features |
|---|---|
| text | length, number of tokens, language, share of non-ASCII characters, empty documents |
| audio | duration, sample rate, number of channels, loudness, silence ratio |
| video | duration, frame rate, resolution, codec |
| embeddings | dimension, norm, NaN values |

---

## References

[1] https://docs.greatexpectations.io/docs/core/introduction/try_gx/

[2] [Expectations Gallery](https://greatexpectations.io/expectations/)

[3] [Pillow: Image module](https://pillow.readthedocs.io/en/stable/reference/Image.html)

[4] [DVC dataset registry](https://github.com/treeverse/dataset-registry)